In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/Documentation_template.md
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/README.md
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/utils/validate_submission.py
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/test/test_source2.tsv
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/test/test_source3.tsv
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/test/test_source1.tsv
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/train/train_ground_truth.tsv
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/train/train_source3.tsv
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/train/train_source2.tsv
/kaggl

# Amazon ML Challenge 2026 — Business Entity Resolution

This notebook develops a business entity resolution pipeline for the Amazon ML Challenge 2026.

The objective is to identify Source 2 and Source 3 records that refer to the same real-world business as each Source 1 entity.

We begin with exploratory data analysis to understand:

- Dataset size and structure
- Country distributions
- Missing and duplicate records
- Business name and address characteristics
- Ground-truth matching patterns
- Matching difficulty and noise

These findings will guide the candidate-generation and matching strategy.

1. How many records are in S1, S2, S3?
2. How much data is in each country?
3. How many duplicate/near-duplicate names and addresses exist?
4. How noisy are the fields?
5. How many matches does each S1 entity have?
6. What percentage of S1 entities are singletons?
7. Are S2/S3 matching patterns different?
8. How often do exact normalized name/address matches already solve the problem?
9. How many candidates would different blocking strategies generate?
10. Most importantly: what does the training ground truth tell us about the actual matching difficulty?

## 1. Environment & Dataset Setup

First, inspect the Kaggle input directory and locate the competition dataset.

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:10]:
        print(f"{indent}    {file}")

input/
    datasets/
        archangel26/
            infinity8-amazon-ml-challenge/
                student_resource/
                    Documentation_template.md
                    README.md
                    utils/
                        validate_submission.py
                    dataset/
                        test/
                            test_source2.tsv
                            test_source3.tsv
                            test_source1.tsv
                        train/
                            train_ground_truth.tsv
                            train_source3.tsv
                            train_source2.tsv
                            train_source1.tsv


### Define Dataset Paths

The competition files are stored inside the `student_resource` directory.

In [ ]:
from pathlib import Path

ROOT = Path(
    "/kaggle/input/datasets/archangel26/"
    "infinity8-amazon-ml-challenge/student_resource"
)

TRAIN_DIR = ROOT / "dataset" / "train"
TEST_DIR = ROOT / "dataset" / "test"

print("Dataset root :", ROOT)
print("Train path   :", TRAIN_DIR)
print("Test path    :", TEST_DIR)

Dataset root : /kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource
Train path   : /kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/train
Test path    : /kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/test


### Verify Dataset Files and Sizes

Check that all expected files are available and inspect their sizes before loading them.

In [ ]:
print("===== TRAIN FILES =====")

for f in sorted(TRAIN_DIR.iterdir()):
    print(f"{f.name:<30} {f.stat().st_size / 1024**2:>10.2f} MB")

print("\n===== TEST FILES =====")

for f in sorted(TEST_DIR.iterdir()):
    print(f"{f.name:<30} {f.stat().st_size / 1024**2:>10.2f} MB")

===== TRAIN FILES =====
train_ground_truth.tsv             121.13 MB
train_source1.tsv                  200.34 MB
train_source2.tsv                  466.63 MB
train_source3.tsv                  480.37 MB

===== TEST FILES =====
test_source1.tsv                   166.91 MB
test_source2.tsv                   485.86 MB
test_source3.tsv                   482.56 MB


## 2. Load Training Data

All competition files are TSV files. We therefore explicitly specify `sep="\t"` when loading them.

The training set contains:

- Source 1
- Source 2
- Source 3
- Ground-truth Source 1 → Source 2/Source 3 mappings

In [ ]:
import pandas as pd
import numpy as np

train_s1 = pd.read_csv(
    TRAIN_DIR / "train_source1.tsv",
    sep="\t",
    dtype=str
)

train_s2 = pd.read_csv(
    TRAIN_DIR / "train_source2.tsv",
    sep="\t",
    dtype=str
)

train_s3 = pd.read_csv(
    TRAIN_DIR / "train_source3.tsv",
    sep="\t",
    dtype=str
)

ground_truth = pd.read_csv(
    TRAIN_DIR / "train_ground_truth.tsv",
    sep="\t",
    dtype=str
)

print("Training data loaded successfully.")

Training data loaded successfully.


## 3. Dataset Overview

Inspect the number of records and columns in each training dataset.

In [ ]:
datasets = {
    "Source 1": train_s1,
    "Source 2": train_s2,
    "Source 3": train_s3,
    "Ground Truth": ground_truth
}

for name, df in datasets.items():
    print(
        f"{name:<15}: "
        f"{df.shape[0]:,} rows × {df.shape[1]} columns"
    )

Source 1       : 2,206,821 rows × 4 columns
Source 2       : 5,034,616 rows × 4 columns
Source 3       : 5,285,603 rows × 4 columns
Ground Truth   : 2,206,821 rows × 2 columns


### Inspect Dataset Columns and Sample Records

Review the schema and a few representative rows from each source.

In [ ]:
for name, df in datasets.items():
    print(f"\n{'=' * 20} {name} {'=' * 20}")
    print("Columns:", list(df.columns))
    display(df.head(3))


==================== Source 1 ====================
Columns: ['entity_id', 'business_name', 'business_address', 'country']


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US



==================== Source 2 ====================
Columns: ['entity_id', 'business_name', 'business_address', 'country']


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India



==================== Source 3 ====================
Columns: ['entity_id', 'business_name', 'business_address', 'country']


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US



==================== Ground Truth ====================
Columns: ['source1_entity_id', 'matched_entity_ids']


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"


## 4. Data Quality Analysis

Before designing the matching algorithm, examine missing values and duplicate records.

Missing fields can make matching more difficult, while duplicates may explain why one Source 1 entity can have multiple corresponding records.

### Missing Values

In [ ]:
for name, df in datasets.items():
    print(f"\n===== {name} =====")

    missing = df.isna().sum()

    missing_table = pd.DataFrame({
        "missing_count": missing,
        "missing_percentage": (
            missing / len(df) * 100
        ).round(2)
    })

    display(missing_table)


===== Source 1 =====


,missing_count,missing_percentage
entity_id,0,0.0
business_name,0,0.0
business_address,0,0.0
country,0,0.0



===== Source 2 =====


,missing_count,missing_percentage
entity_id,0,0.00
business_name,2,0.00
business_address,168967,3.36
country,0,0.00



===== Source 3 =====


,missing_count,missing_percentage
entity_id,0,0.00
business_name,13,0.00
business_address,175916,3.33
country,0,0.00



===== Ground Truth =====


,missing_count,missing_percentage
source1_entity_id,0,0.00
matched_entity_ids,123247,5.58


### Exact Duplicate Records

Check for completely duplicated rows within each source.

In [ ]:
for name, df in {
    "Source 1": train_s1,
    "Source 2": train_s2,
    "Source 3": train_s3
}.items():

    duplicate_count = df.duplicated().sum()

    print(
        f"{name:<10}: "
        f"{duplicate_count:,} duplicate rows "
        f"({duplicate_count / len(df) * 100:.2f}%)"
    )

Source 1  : 0 duplicate rows (0.00%)
Source 2  : 0 duplicate rows (0.00%)
Source 3  : 0 duplicate rows (0.00%)


## 5. Country Distribution

Analyze the country distribution across Source 1, Source 2, and Source 3.

Country can be used as a strong matching feature, but the model should not assume that only the training countries will appear at test time because France is introduced in the test data.

In [ ]:
for name, df in {
    "Source 1": train_s1,
    "Source 2": train_s2,
    "Source 3": train_s3
}.items():

    print(f"\n===== {name} =====")

    country_counts = (
        df["country"]
        .fillna("<MISSING>")
        .value_counts()
    )

    country_table = pd.DataFrame({
        "count": country_counts,
        "percentage": (
            country_counts / len(df) * 100
        ).round(2)
    })

    display(country_table)


===== Source 1 =====


,count,percentage
country,,
US,1323633,59.98
India,883188,40.02



===== Source 2 =====


,count,percentage
country,,
US,3016817,59.92
India,2017799,40.08



===== Source 3 =====


,count,percentage
country,,
US,3170056,59.98
India,2115547,40.02


## 6. Business Name Analysis

Business names are expected to contain:

- Abbreviations
- Legal suffix variations
- Punctuation differences
- Typos
- Transliteration
- Different word ordering
- `&` versus `and`

We first inspect their basic characteristics.

## 6. Business Name Analysis

Business names are expected to contain:

- Abbreviations
- Legal suffix variations
- Punctuation differences
- Typos
- Transliteration
- Different word ordering
- `&` versus `and`

We first inspect their basic characteristics.

In [ ]:
for name, df in {
    "Source 1": train_s1,
    "Source 2": train_s2,
    "Source 3": train_s3
}.items():

    lengths = df["business_name"].fillna("").str.len()

    print(f"\n===== {name} =====")
    print(lengths.describe().round(2))


===== Source 1 =====
count    2206821.00
mean          24.03
std            7.74
min            3.00
25%           18.00
50%           24.00
75%           30.00
max          105.00
Name: business_name, dtype: float64

===== Source 2 =====
count    5034616.00
mean          25.10
std            8.89
min            0.00
25%           19.00
50%           25.00
75%           31.00
max          104.00
Name: business_name, dtype: float64

===== Source 3 =====
count    5285603.00
mean          25.20
std            9.49
min            0.00
25%           18.00
50%           25.00
75%           31.00
max          123.00
Name: business_name, dtype: float64


## 7. Business Address Analysis

Addresses may contain:

- Abbreviations
- Missing components
- Reordered components
- Transliteration
- Municipal numbering differences
- Landmark-based descriptions

We inspect basic address characteristics before designing address normalization.

### Business Address Length Distribution

In [ ]:
for name, df in {
    "Source 1": train_s1,
    "Source 2": train_s2,
    "Source 3": train_s3
}.items():

    lengths = df["business_address"].fillna("").str.len()

    print(f"\n===== {name} =====")
    print(lengths.describe().round(2))


===== Source 1 =====
count    2206821.00
mean          52.07
std           25.33
min           11.00
25%           33.00
50%           41.00
75%           70.00
max          256.00
Name: business_address, dtype: float64

===== Source 2 =====
count    5034616.00
mean          46.23
std           24.84
min            0.00
25%           30.00
50%           37.00
75%           61.00
max          249.00
Name: business_address, dtype: float64

===== Source 3 =====
count    5285603.00
mean          46.71
std           21.65
min            0.00
25%           35.00
50%           42.00
75%           54.00
max          240.00
Name: business_address, dtype: float64


## 8. Ground Truth Analysis

The ground truth provides the actual Source 2 and Source 3 records corresponding to each Source 1 entity.

This is the most important part of the training data because the competition allows:

- Zero matches
- One match
- Multiple matches

Understanding this distribution is essential for designing a precision-focused matcher.

### Number of Matches per Source 1 Entity

In [ ]:
gt = ground_truth.copy()

gt["matched_entity_ids"] = (
    gt["matched_entity_ids"]
    .fillna("")
)

gt["num_matches"] = gt["matched_entity_ids"].apply(
    lambda x: 0 if not x.strip() else len(x.split(","))
)

print("Match-count statistics:")
display(gt["num_matches"].describe())

Match-count statistics:


count    2.206821e+06
mean     3.461253e+00
std      1.705323e+00
min      0.000000e+00
25%      2.000000e+00
50%      3.000000e+00
75%      5.000000e+00
max      1.100000e+01
Name: num_matches, dtype: float64

### Match Count Distribution

In [ ]:
match_distribution = (
    gt["num_matches"]
    .value_counts()
    .sort_index()
    .rename_axis("number_of_matches")
    .reset_index(name="source1_entities")
)

match_distribution["percentage"] = (
    match_distribution["source1_entities"]
    / len(gt)
    * 100
).round(2)

display(match_distribution)

,number_of_matches,source1_entities,percentage
0,0,123247,5.58
1,1,119157,5.40
2,2,375212,17.00
3,3,530841,24.05
4,4,484115,21.94
5,5,321957,14.59
6,6,164868,7.47
7,7,63968,2.90
8,8,18680,0.85
9,9,4205,0.19


### Zero, Single, and Multiple Match Rates

The F0.5 metric is evaluated per Source 1 entity, so understanding the proportion of unmatched and multi-match entities is particularly important.

In [ ]:
zero_matches = (gt["num_matches"] == 0).sum()
one_match = (gt["num_matches"] == 1).sum()
multiple_matches = (gt["num_matches"] > 1).sum()

print(
    f"Zero matches     : {zero_matches:,} "
    f"({zero_matches / len(gt) * 100:.2f}%)"
)

print(
    f"Exactly one      : {one_match:,} "
    f"({one_match / len(gt) * 100:.2f}%)"
)

print(
    f"Multiple matches : {multiple_matches:,} "
    f"({multiple_matches / len(gt) * 100:.2f}%)"
)

Zero matches     : 123,247 (5.58%)
Exactly one      : 119,157 (5.40%)
Multiple matches : 1,964,417 (89.02%)


## 9. Source 2 vs Source 3 Matching Patterns

Determine whether Source 1 entities tend to match Source 2, Source 3, or both.

This helps determine whether we should use separate candidate-generation and scoring strategies for the two sources.

In [ ]:
def classify_match_sources(match_string):
    if not match_string.strip():
        return "none"

    ids = [x.strip() for x in match_string.split(",")]

    has_s2 = any(x.startswith("S2") for x in ids)
    has_s3 = any(x.startswith("S3") for x in ids)

    if has_s2 and has_s3:
        return "S2 + S3"
    elif has_s2:
        return "S2 only"
    elif has_s3:
        return "S3 only"
    else:
        return "unknown"


gt["match_source_type"] = (
    gt["matched_entity_ids"]
    .apply(classify_match_sources)
)

source_distribution = (
    gt["match_source_type"]
    .value_counts()
    .rename_axis("match_source_type")
    .reset_index(name="source1_entities")
)

source_distribution["percentage"] = (
    source_distribution["source1_entities"]
    / len(gt)
    * 100
).round(2)

display(source_distribution)

,match_source_type,source1_entities,percentage
0,S2 + S3,1776047,80.48
1,S3 only,164498,7.45
2,S2 only,143029,6.48
3,none,123247,5.58


## 10. Inspect Ground-Truth Examples

Inspect individual Source 1 entities and their known matches to qualitatively understand the types of noise present in the data.

In [ ]:
display(
    gt[
        [
            "source1_entity_id",
            "matched_entity_ids",
            "num_matches",
            "match_source_type"
        ]
    ].sample(
        min(20, len(gt)),
        random_state=42
    )
)

,source1_entity_id,matched_entity_ids,num_matches,match_source_type
2195840,S1-384734976,"S2-685146894,S2-908470079,S3-109845102,S3-1764...",4,S2 + S3
1487051,S1-574038342,"S2-133116927,S2-180265553,S2-201976425",3,S2 only
1878277,S1-189408085,,0,none
993966,S1-797774718,"S2-151469064,S3-994706509",2,S2 + S3
565428,S1-729164458,"S2-619382517,S2-949886625,S3-532795614",3,S2 + S3
1645753,S1-102324451,"S2-626433724,S3-583628765,S3-900962957",3,S2 + S3
1165724,S1-897032402,"S2-119959444,S2-737316263,S2-487624450,S2-4866...",6,S2 + S3
363842,S1-345344463,"S2-731601561,S3-474125535",2,S2 + S3
962445,S1-281641466,"S2-559704787,S3-772912089",2,S2 + S3
964820,S1-608038725,"S2-895472957,S2-192681436,S2-217803628,S3-9935...",4,S2 + S3


# 11. Matching-Specific Exploratory Data Analysis

The basic dataset exploration is now complete.

We next investigate the actual entity-resolution problem:

1. How effective is normalization?
2. How many true matches have identical normalized names?
3. How many have identical normalized addresses?
4. How much ambiguity exists after normalization?
5. How many candidates are generated by different blocking strategies?
6. How much of the ground truth is recovered by each strategy?

These measurements will determine the final candidate-generation and matching architecture.

```
S1: 2,206,821
S2: 5,034,616
S3: 5,285,603
Ground truth: 2,206,821
S2 address missing: 3.36%
S3 address missing: 3.33%
Exact duplicate full rows: 0%
Training countries: ~60% US / 40% India
5.58% of S1 have no match
5.40% have exactly one match
89.02% have multiple matches
80.48% match both S2 and S3

```

## 12. Conservative Text Normalization

Business records may differ because of capitalization, punctuation, accents, whitespace, and representations such as `&` versus `and`.

We create normalized versions of names and addresses while preserving the original columns.

The original TSV files are not modified.

In [ ]:
import re
import unicodedata


def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()

    # Unicode normalization
    text = unicodedata.normalize("NFKD", text)

    # Remove accent marks
    text = "".join(
        char for char in text
        if not unicodedata.combining(char)
    )

    # Normalize ampersand
    text = text.replace("&", " and ")

    # Replace punctuation with spaces
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Collapse repeated whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_name(text):
    return normalize_text(text)


def normalize_address(text):
    return normalize_text(text)

## 13. Apply Conservative Normalization

Create `name_norm` and `address_norm` columns for the three training sources.

These are additional in-memory columns and do not alter the original competition files.

In [ ]:
for df in [train_s1, train_s2, train_s3]:

    df["name_norm"] = (
        df["business_name"]
        .fillna("")
        .apply(normalize_name)
    )

    df["address_norm"] = (
        df["business_address"]
        .fillna("")
        .apply(normalize_address)
    )

print("Normalization complete.")

display(
    train_s1[
        [
            "business_name",
            "name_norm",
            "business_address",
            "address_norm"
        ]
    ].head(10)
)

Normalization complete.


,business_name,name_norm,business_address,address_norm
0,Orelee's Barbershop,orelee s barbershop,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc
1,Prime Money,prime money,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok
2,B+ Retail Inc,b retail inc,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az
3,Christ Chapel,christ chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md
4,Prabhav Business Center,prabhav business center,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal
5,Custom Wealth Services LLC,custom wealth services llc,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue
6,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...
7,Nexus Anchor Rain,nexus anchor rain,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn
8,Moore Bitwise Inc,moore bitwise inc,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in
9,Dermatology Green Medicine,dermatology green medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...


## 14. Normalized Business Name Uniqueness

Before using exact normalized names for blocking, we need to know how ambiguous they are.

A normalized name that occurs thousands of times is not a useful standalone blocking key, while a rare normalized name can be highly informative.

In [ ]:
for name, df in {
    "Source 1": train_s1,
    "Source 2": train_s2,
    "Source 3": train_s3
}.items():

    counts = df["name_norm"].value_counts()

    non_empty = counts[counts.index != ""]

    print(f"\n===== {name} =====")
    print(f"Unique normalized names      : {len(non_empty):,}")
    print(
        f"Names occurring more than once: "
        f"{(non_empty > 1).sum():,}"
    )
    print(
        f"Maximum name frequency       : "
        f"{non_empty.max():,}"
    )


===== Source 1 =====
Unique normalized names      : 1,522,166
Names occurring more than once: 180,374
Maximum name frequency       : 253

===== Source 2 =====
Unique normalized names      : 3,751,586
Names occurring more than once: 362,196
Maximum name frequency       : 1,672

===== Source 3 =====
Unique normalized names      : 4,048,679
Names occurring more than once: 393,930
Maximum name frequency       : 2,957


## 15. Normalized Address Uniqueness

Perform the same analysis for normalized addresses.

Addresses can be powerful matching signals, but shared addresses may correspond to multiple businesses.

In [ ]:
for name, df in {
    "Source 1": train_s1,
    "Source 2": train_s2,
    "Source 3": train_s3
}.items():

    counts = df["address_norm"].value_counts()

    non_empty = counts[counts.index != ""]

    print(f"\n===== {name} =====")
    print(f"Unique normalized addresses       : {len(non_empty):,}")
    print(
        f"Addresses occurring more than once: "
        f"{(non_empty > 1).sum():,}"
    )
    print(
        f"Maximum address frequency        : "
        f"{non_empty.max():,}"
    )


===== Source 1 =====
Unique normalized addresses       : 2,130,152
Addresses occurring more than once: 40,530
Maximum address frequency        : 14

===== Source 2 =====
Unique normalized addresses       : 4,281,378
Addresses occurring more than once: 461,194
Maximum address frequency        : 19

===== Source 3 =====
Unique normalized addresses       : 4,614,584
Addresses occurring more than once: 393,879
Maximum address frequency        : 35


## 16. Inspect Normalization Examples

Inspect randomly selected business names and addresses before and after normalization.

This helps verify that normalization removes superficial variation without destroying useful information.

In [ ]:
sample_s1 = train_s1[
    [
        "business_name",
        "name_norm",
        "business_address",
        "address_norm"
    ]
].sample(
    min(30, len(train_s1)),
    random_state=42
)

display(sample_s1)

,business_name,name_norm,business_address,address_norm
2195840,Pediatric Medicine PLLC,pediatric medicine pllc,"4850 20, Otisco, NY",4850 20 otisco ny
1487051,Fetech National Twin,fetech national twin,"19034 Woodburn Road, Woodburn, IN",19034 woodburn road woodburn in
1878277,General Design Innovations LLC,general design innovations llc,"7241 Osage Avenue, Mesa, AZ",7241 osage avenue mesa az
993966,Construction Ideaz Papers Private Limited,construction ideaz papers private limited,"Building No.4/606, Prabhul Cottage Karimbalur,...",building no 4 606 prabhul cottage karimbalur e...
565428,Penaloza and Bittle First Inc.,penaloza and bittle first inc,"4255 Charleswood Avenue, Memphis, TN",4255 charleswood avenue memphis tn
1645753,National Investments LLC,national investments llc,"1641 Virginia Lane, Hueytown, AL",1641 virginia lane hueytown al
1165724,Reus Idaho Company,reus idaho company,"Sanford, ME, 31 Guillemette Street",sanford me 31 guillemette street
363842,Creative Projects Limited,creative projects limited,"12Thfloor, C, 1204, Roya Oasis, Jankalyan Naga...",12thfloor c 1204 roya oasis jankalyan nagarmal...
962445,Goar Stone LLC,goar stone llc,"2105 Wood Ridge Cove, Cedar Park, TX",2105 wood ridge cove cedar park tx
964820,Lakshmi Consultants Private Limited,lakshmi consultants private limited,"A-301, New Sai Dham Chsl, Ramdev Park Road, Th...",a 301 new sai dham chsl ramdev park road thane...


## 17. Construct Ground-Truth Match Pairs

The ground truth stores matched Source 2 and Source 3 entity IDs as a comma-separated list.

We convert this representation into individual Source 1 → matched entity pairs so that we can directly compare the true matches against candidate-generation strategies.

In [ ]:
# Create lookup tables for entity IDs
s2_lookup = train_s2.set_index("entity_id")
s3_lookup = train_s3.set_index("entity_id")

# Keep only non-empty ground-truth mappings
gt_nonempty = gt[
    gt["matched_entity_ids"].str.strip() != ""
].copy()

# Count individual true matches
total_true_matches = (
    gt_nonempty["matched_entity_ids"]
    .str.split(",")
    .explode()
    .shape[0]
)

print(f"Source 1 entities with at least one match: {len(gt_nonempty):,}")
print(f"Total ground-truth matched pairs: {total_true_matches:,}")

Source 1 entities with at least one match: 2,083,574
Total ground-truth matched pairs: 7,638,365


## 18. Exact Normalized Name Matching

We now test one of the simplest possible matching signals:

`country + normalized business name`

For each ground-truth match, we check whether the matched record has exactly the same normalized business name as the Source 1 record.

This measures how much of the problem can be solved using exact normalized names alone.

In [ ]:
# Create entity -> normalized field mappings
s1_name = train_s1.set_index("entity_id")["name_norm"]
s2_name = train_s2.set_index("entity_id")["name_norm"]
s3_name = train_s3.set_index("entity_id")["name_norm"]

s1_country = train_s1.set_index("entity_id")["country"]
s2_country = train_s2.set_index("entity_id")["country"]
s3_country = train_s3.set_index("entity_id")["country"]

name_match_count = 0
valid_true_pairs = 0

for row in gt_nonempty.itertuples(index=False):

    s1_id = row.source1_entity_id
    s1_name_value = s1_name.get(s1_id, "")
    s1_country_value = s1_country.get(s1_id, "")

    if not s1_name_value:
        continue

    for matched_id in row.matched_entity_ids.split(","):
        matched_id = matched_id.strip()

        if matched_id.startswith("S2"):
            candidate_name = s2_name.get(matched_id, "")
            candidate_country = s2_country.get(matched_id, "")
        else:
            candidate_name = s3_name.get(matched_id, "")
            candidate_country = s3_country.get(matched_id, "")

        valid_true_pairs += 1

        if (
            s1_name_value == candidate_name
            and s1_country_value == candidate_country
        ):
            name_match_count += 1

print(f"True pairs evaluated : {valid_true_pairs:,}")
print(f"Exact normalized name matches : {name_match_count:,}")
print(
    f"Recall : "
    f"{name_match_count / valid_true_pairs * 100:.2f}%"
)

True pairs evaluated : 7,638,365
Exact normalized name matches : 1,958,580
Recall : 25.64%


## 19. Exact Normalized Address Matching

Repeat the same experiment using:

`country + normalized address`

This tells us how much additional matching information is available from addresses.

In [ ]:
s1_address = train_s1.set_index("entity_id")["address_norm"]
s2_address = train_s2.set_index("entity_id")["address_norm"]
s3_address = train_s3.set_index("entity_id")["address_norm"]

address_match_count = 0
valid_address_pairs = 0

for row in gt_nonempty.itertuples(index=False):

    s1_id = row.source1_entity_id
    s1_address_value = s1_address.get(s1_id, "")
    s1_country_value = s1_country.get(s1_id, "")

    if not s1_address_value:
        continue

    for matched_id in row.matched_entity_ids.split(","):
        matched_id = matched_id.strip()

        if matched_id.startswith("S2"):
            candidate_address = s2_address.get(matched_id, "")
            candidate_country = s2_country.get(matched_id, "")
        else:
            candidate_address = s3_address.get(matched_id, "")
            candidate_country = s3_country.get(matched_id, "")

        valid_address_pairs += 1

        if (
            s1_address_value == candidate_address
            and s1_country_value == candidate_country
        ):
            address_match_count += 1

print(f"True pairs evaluated : {valid_address_pairs:,}")
print(f"Exact normalized address matches : {address_match_count:,}")
print(
    f"Recall : "
    f"{address_match_count / valid_address_pairs * 100:.2f}%"
)

True pairs evaluated : 7,638,365
Exact normalized address matches : 632,221
Recall : 8.28%
